# HAETAE-IRV: 전체 서명 결함주입 실험 (STM32F4, CW-Husky)

**위에서부터 순서대로** 실행. 모든 함수는 SETUP의 한 셀에 모아 정의(커널 재시작에 강함).

- 변형: `baseline`/`double`/`leeha`/`irv`. 클럭: **HSE 직결 7.37MHz**(글리치 가능).
- 결함지점(슈도라인): UNPACK(1)·SEED(2)·SIGNBIT(3)·LSB(5)·CS=T1(7)·ADDY=T2(8)·REJECT=RB(9).
- 커맨드: `e`echo/`k`keygen/`p`서명(16B=메시지)/`t`사이클 · FAULT_SIM: `f`[라인,oneshot,checkskip]/`x`z1/`s`참s1/`c`챌린지.

**순서**: SETUP → BUILD → SMOKE → EXP1(T2직접) → EXP2(커버리지) → EXP3(재현율) → EXP4(2차결함) → EXP5(오버헤드) → EXP6(연산시간).

## SETUP

In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM  = 'CW308_STM32F4'
CRYPTO_TARGET = 'NONE'
SS_VER = 'SS_VER_1_1'

In [2]:
%matplotlib inline
import chipwhisperer as cw
import numpy as np, matplotlib.pyplot as plt
import time, struct, csv, collections, statistics
scope = cw.scope(name='Husky')

In [3]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen            

In [4]:
%%bash
cd ../../../
ls firmware/mcu/hal/chipwhisperer-fw-extra/stm32f4/Makefile.stm32f4 >/dev/null 2>&1 && echo 'F4 HAL OK' || git submodule update --init firmware/mcu/hal/chipwhisperer-fw-extra

F4 HAL OK


In [5]:
# ===== 클럭 + 모든 헬퍼/함수 정의 (이 셀 하나로 전부) =====
scope.clock.clkgen_freq = 7.37e6
scope.clock.adc_mul = 1
scope.io.hs2 = 'clkgen'
time.sleep(0.2)
print('clkgen_freq =', scope.clock.clkgen_freq, ' hs2 =', scope.io.hs2)

import importlib, haetae_recover; importlib.reload(haetae_recover)
from haetae_recover import attack_recover

FW = '../../../firmware/mcu/simpleserial-haetae/'
FL = {'NONE':0,'SEED':1,'SIGNBIT':2,'UNPACK':3,'LSB':4,'CS':5,'ADDY':6,'REJECT':7}
FAULT_LINES = ['NONE','SEED','SIGNBIT','UNPACK','LSB','CS','ADDY','REJECT']
RESULTS = collections.OrderedDict()

def flash(hexname):
    cw.program_target(scope, prog, FW + hexname)
    reset_target(scope); time.sleep(0.5); target.flush()
def recover_target():
    reset_target(scope); time.sleep(0.5); target.flush()
def ss_echo(timeout=3000):
    target.flush(); target.simpleserial_write('e', bytearray())
    r = target.simpleserial_read('r', 16, timeout=timeout); return bytes(r) if r else None
def ss_keygen(timeout=60000):
    target.flush(); target.simpleserial_write('k', bytearray())
    r = target.simpleserial_read('r', 1, timeout=timeout); return bytes(r) if r else None
def ss_sign_msg(msg16, timeout=90000):
    target.flush(); target.simpleserial_write('p', bytearray(msg16))
    r = target.simpleserial_read('r', 16, timeout=timeout); return bytes(r) if r else None
def ss_sign(timeout=90000):
    return ss_sign_msg([0]*16, timeout)
def ss_cycles(timeout=120000):
    target.flush(); target.simpleserial_write('t', bytearray([0]*16))
    r = target.simpleserial_read('r', 4, timeout=timeout); return struct.unpack('<I', bytes(r))[0] if r else None
def set_fault(line, oneshot=0, checkskip=0):
    target.flush(); target.simpleserial_write('f', bytes([line, oneshot, checkskip]))
    return target.simpleserial_read('r', 1, timeout=3000)

def sweep_variant(label, recover_t2=False, timeout=90000):
    print('[%s]' % label); res = {}
    for name in FAULT_LINES:
        set_fault(FL[name], 0)
        d = ss_sign(timeout=timeout); res[name] = d.hex() if d else None
        tag = ''
        if d is None: recover_target()
        elif recover_t2 and name == 'ADDY':
            try: tag = ' | T2 s1=%.0f%%' % (100*attack_recover(target)['agreement'])
            except Exception: tag = ' | recover_err'
        print('   %-8s %s%s' % (name, res[name], tag))
    RESULTS[label] = res; return res

def measure_success(faults, M=10, reps=3, verify_t2=('ADDY',), timeout=90000):
    msgs = [[(j & 0xFF), ((j >> 8) & 0xFF)] + [0]*14 for j in range(M*reps)]
    clean = {}
    for j, msg in enumerate(msgs):
        set_fault(FL['NONE'], 0); d = ss_sign_msg(msg, timeout)
        if d is None: recover_target(); set_fault(FL['NONE'],0); d = ss_sign_msg(msg, timeout)
        clean[j] = d.hex() if d else None
    print('clean 캐시 %d/%d' % (sum(v is not None for v in clean.values()), len(msgs)))
    rows = collections.OrderedDict()
    for fn in faults:
        brate, bkey = [], []
        for b in range(reps):
            man = key = n = 0
            for i in range(M):
                j = b*M + i
                if clean[j] is None: continue
                n += 1; set_fault(FL[fn], 1); d = ss_sign_msg(msgs[j], timeout)
                if d is None: recover_target(); continue
                if d.hex() != clean[j]:
                    man += 1
                    if fn in verify_t2:
                        try:
                            r = attack_recover(target)
                            if r['clean_T2'] and r['agreement'] == 1.0: key += 1
                        except Exception: pass
            brate.append(100.0*man/max(n,1))
            if fn in verify_t2: bkey.append(100.0*key/max(n,1))
        r = statistics.mean(brate); rows[fn] = {'rate': r, 'batches': brate}
        att = ('%.1f' % (100.0/r)) if r > 0 else 'inf'
        line = '%-8s 재현율 %.1f%% %s | ~%s회/누설' % (fn, r, ['%.0f' % x for x in brate], att)
        if fn in verify_t2:
            k = statistics.mean(bkey); rows[fn]['key'] = k; line += ' | 키복원 %.1f%% of %s' % (k, ['%.0f'%x for x in bkey])
        print(line)
    return rows

print('defs OK — 모든 함수 정의 완료')

(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:704) Target clock may drop; you may need to reset your target.


clkgen_freq = 7371428.571428572  hs2 = clkgen
defs OK — 모든 함수 정의 완료


## BUILD — 8개 hex 한 번에 (clean 4 + FAULT_SIM 4). ~3분

In [6]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../firmware/mcu/simpleserial-haetae
for V in baseline double leeha irv; do
  make clean PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 VARIANT=$V >/dev/null 2>&1
  make       PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 VARIANT=$V >/dev/null 2>&1 && cp simpleserial-haetae-$1.hex haetae-$V-$1.hex
  make clean PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 VARIANT=$V FAULT_SIM=1 >/dev/null 2>&1
  make       PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 VARIANT=$V FAULT_SIM=1 >/dev/null 2>&1 && cp simpleserial-haetae-$1.hex haetae-$V-FSIM-$1.hex
  echo "built $V (clean + FSIM)"
done

built baseline (clean + FSIM)
built double (clean + FSIM)
built leeha (clean + FSIM)
built irv (clean + FSIM)


## SMOKE — baseline 통신 + GOLDEN + 사이클

In [7]:
flash('haetae-baseline-{}.hex'.format(PLATFORM))
print('echo  :', (ss_echo() or b'').hex(), ' (a0a1..af 기대)')
print('keygen:', (ss_keygen() or b'').hex(), ' (00 기대)')
g1 = ss_sign(); g2 = ss_sign()
print('sign#1:', g1.hex() if g1 else None)
print('sign#2:', g2.hex() if g2 else None)
assert g1 and g1 == g2, '재현성 실패'
GOLDEN = g1
print('GOLDEN(F4) =', GOLDEN.hex())
print('baseline sign cycles =', ss_cycles())

Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 25123 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 25123 bytes
echo  : a0a1a2a3a4a5a6a7a8a9aaabacadaeaf  (a0a1..af 기대)
keygen: 00  (00 기대)
sign#1: ba9f152c607b207fc6512635ba11388c
sign#2: ba9f152c607b207fc6512635ba11388c
GOLDEN(F4) = ba9f152c607b207fc6512635ba11388c
baseline sign cycles = 60858575


## EXP1 — T2 직접 추출 (baseline FSIM): 단일 서명 s1 100% 복원

In [8]:
flash('haetae-baseline-FSIM-{}.hex'.format(PLATFORM))
set_fault(FL['NONE']); g = ss_sign(); print('FL_NONE :', g.hex() if g else None, '(=GOLDEN)')
set_fault(FL['ADDY']); d = ss_sign(); print('FL_ADDY :', d.hex() if d else None, '(T2 누설)')
res = attack_recover(target)
print('clean_T2 =', res['clean_T2'], '| s1 일치율 = %.1f%%' % (100*res['agreement']), '| sign', res['sign'])

Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 25683 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 25683 bytes
FL_NONE : ba9f152c607b207fc6512635ba11388c (=GOLDEN)
FL_ADDY : 63ff5ebfaa6263739651890939cccb48 (T2 누설)
clean_T2 = True | s1 일치율 = 100.0% | sign -


## EXP2 — 커버리지 (4변형 × 7결함, 1메시지) → 표 A
`golden`=무영향 · `LEAK`=baseline과 동일(미차단) · `blocked`=무효화/난수화. baseline FL_ADDY는 EXP1에서 s1 100%로 'LEAK' 실효성 입증.

In [9]:
RESULTS.clear()
flash('haetae-baseline-FSIM-{}.hex'.format(PLATFORM)); sweep_variant('baseline', recover_t2=True)
for V in ['double','leeha','irv']:
    flash('haetae-{}-FSIM-{}.hex'.format(V, PLATFORM)); print('echo:', (ss_echo() or b'').hex()[:8])
    sweep_variant(V)

Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 25683 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 25683 bytes
[baseline]
   NONE     ba9f152c607b207fc6512635ba11388c
   SEED     7d68d2845d9bce2ceef3591fda4583a7
   SIGNBIT  d6c977b847eeb9f2c30ac1ab4a8fa4c5
   UNPACK   99f2b8e52e112a5863677bdfde153837
   LSB      4f6bb1ef6098c2c9abbba07930f73bb3
   CS       9b46b88b2dfcbf53169ed3a778024414
   ADDY     63ff5ebfaa6263739651890939cccb48 | T2 s1=100%
   REJECT   ab4a71a7f8084e289653c64b8105a54d
Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 25891 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 25891 bytes
echo: a0a1a2a3
[double]
   NONE     ba9f152c607b207fc6512635ba11388c
   SEED     a70948f5783c58a8596fa54e76dd1e9a
   SIGNBIT  a70948f5783c58a8596fa54e76dd1e9a
 

In [10]:
GH = GOLDEN.hex(); base = RESULTS['baseline']
def classify(label, name):
    d = RESULTS[label].get(name)
    if d is None:           return 'MUTE'
    if d == GH:             return 'golden'
    if d == base.get(name): return 'LEAK'
    return 'blocked'
cols = list(RESULTS.keys())
hdr = '%-9s | ' % 'fault' + ' | '.join('%-8s' % c for c in cols); print(hdr); print('-'*len(hdr))
for name in FAULT_LINES:
    if name == 'NONE': continue
    print('%-9s | ' % name + ' | '.join('%-8s' % classify(c, name) for c in cols))
print('\nFL_NONE 오탐:', {c: ('OK' if RESULTS[c].get('NONE')==GH else 'FAIL') for c in cols})
with open('haetae_f4_coverage.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['fault']+cols)
    for name in FAULT_LINES:
        if name=='NONE': continue
        w.writerow([name]+[classify(c,name) for c in cols])
print('saved haetae_f4_coverage.csv')

fault     | baseline | double   | leeha    | irv     
-----------------------------------------------------
SEED      | LEAK     | blocked  | blocked  | blocked 
SIGNBIT   | LEAK     | blocked  | blocked  | blocked 
UNPACK    | LEAK     | blocked  | blocked  | blocked 
LSB       | LEAK     | blocked  | blocked  | blocked 
CS        | LEAK     | blocked  | blocked  | blocked 
ADDY      | LEAK     | blocked  | blocked  | blocked 
REJECT    | LEAK     | blocked  | LEAK     | blocked 

FL_NONE 오탐: {'baseline': 'OK', 'double': 'OK', 'leeha': 'OK', 'irv': 'OK'}
saved haetae_f4_coverage.csv


## EXP3 — 누설 재현율 (baseline, one-shot, 메시지 가변). 빠르게 보려면 M=5

In [11]:
flash('haetae-baseline-FSIM-{}.hex'.format(PLATFORM))
SR = measure_success(['SEED','SIGNBIT','UNPACK','LSB','CS','ADDY','REJECT'], M=10, reps=3)
with open('haetae_f4_success_rate.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['fault','reproduce_%','attempts_per_leak','key_recover_%','batches'])
    for fn,v in SR.items():
        att=(100.0/v['rate']) if v['rate']>0 else None
        w.writerow([fn,'%.1f'%v['rate'],('%.1f'%att) if att else 'inf',('%.1f'%v['key']) if 'key' in v else '',v['batches']])
print('saved haetae_f4_success_rate.csv')

Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 25683 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 25683 bytes
clean 캐시 30/30
SEED     재현율 100.0% ['100', '100', '100'] | ~1.0회/누설
SIGNBIT  재현율 23.3% ['10', '10', '50'] | ~4.3회/누설
UNPACK   재현율 100.0% ['100', '100', '100'] | ~1.0회/누설
LSB      재현율 23.3% ['20', '20', '30'] | ~4.3회/누설
CS       재현율 23.3% ['10', '30', '30'] | ~4.3회/누설
ADDY     재현율 83.3% ['70', '90', '90'] | ~1.2회/누설 | 키복원 83.3% of ['70', '90', '90']
REJECT   재현율 90.0% ['90', '90', '90'] | ~1.1회/누설
saved haetae_f4_success_rate.csv


## EXP4 — 2차 결함 (T2 + 검사분기 스킵) ★ IRV 무분기 우위
detect-and-abort(double 비교·leeha 검증)는 검사분기 스킵으로 우회(LEAK), IRV는 응답연산 핵심을 무분기 감염으로 처리해 차단 유지.

In [12]:
flash('haetae-baseline-FSIM-{}.hex'.format(PLATFORM))
set_fault(FL['ADDY'], 0, 0); LEAK2 = ss_sign(); ref = LEAK2.hex() if LEAK2 else None
print('기준 T2 누설:', ref)
print('\n=== 2차결함 (T2 ADDY + 검사분기 스킵) ===')
SO = collections.OrderedDict()
for V in ['baseline','double','leeha','irv']:
    flash('haetae-{}-FSIM-{}.hex'.format(V, PLATFORM))
    set_fault(FL['ADDY'], 0, 1); d = ss_sign()
    if d is None: recover_target()
    h = d.hex() if d else None
    verdict = 'LEAK(우회)' if h == ref else ('blocked' if h else 'MUTE')
    SO[V] = (h, verdict); print('  %-9s %s -> %s' % (V, h, verdict))
print('\n해석: double/leeha=검사분기 스킵으로 우회(LEAK), IRV=무분기라 차단 유지.')
with open('haetae_f4_2ndorder.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['variant','digest','verdict']); [w.writerow([V,h,v]) for V,(h,v) in SO.items()]
print('saved haetae_f4_2ndorder.csv')

Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 25683 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 25683 bytes
기준 T2 누설: 63ff5ebfaa6263739651890939cccb48

=== 2차결함 (T2 ADDY + 검사분기 스킵) ===
Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 25683 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 25683 bytes
  baseline  63ff5ebfaa6263739651890939cccb48 -> LEAK(우회)
Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 25891 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 25891 bytes
  double    63ff5ebfaa6263739651890939cccb48 -> LEAK(우회)
Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 31723

## EXP5 — 구현 오버헤드 (코드/RAM, 무결함 빌드) → 표 C

In [13]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../firmware/mcu/simpleserial-haetae
printf '%-10s %8s %6s %7s %8s\n' variant text data bss dec
for V in baseline double leeha irv; do
  make clean PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 VARIANT=$V >/dev/null 2>&1
  make       PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 VARIANT=$V >/dev/null 2>&1
  S=$(arm-none-eabi-size simpleserial-haetae-$1.elf | tail -1)
  printf '%-10s %8s %6s %7s %8s\n' $V $(echo $S | awk '{print $1, $2, $3, $4}')
done

variant        text   data     bss      dec
baseline      24556    564    6048    31168
double        24732    564    6048    31344
leeha         30748    564    6056    37368
irv           31004    564    6056    37624


## EXP6 — 연산시간 (서명 1회 DWT 사이클, 무결함 빌드) → 표 C

In [14]:
CYC = collections.OrderedDict()
for V in ['baseline','double','leeha','irv']:
    flash('haetae-{}-{}.hex'.format(V, PLATFORM))
    ss_keygen()
    CYC[V] = ss_cycles(); print('%-9s %s cycles' % (V, CYC[V]))
b = CYC.get('baseline') or 1
print('\n%-9s %-12s %s' % ('변형','사이클','상대'))
for V, c in CYC.items(): print('%-9s %-12s %.3f x' % (V, c, (c/b) if c else float('nan')))
with open('haetae_f4_timing.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['variant','sign_cycles','relative'])
    for V,c in CYC.items(): w.writerow([V, c, ('%.3f'%(c/b)) if c else ''])
print('saved haetae_f4_timing.csv')

Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 25123 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 25123 bytes
baseline  60858575 cycles
Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 25299 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 25299 bytes
double    121717096 cycles
Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 31315 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 31315 bytes
leeha     69175415 cycles
Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 31571 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 31571 bytes
irv      

## 산출 CSV (→ `D:/06_github_desktop/fia_cm_haetae/test/2026-06-30/`)
- `haetae_f4_coverage.csv` (표 A) · `haetae_f4_success_rate.csv` (재현율) · `haetae_f4_2ndorder.csv` (2차결함) · `haetae_f4_timing.csv` + EXP5 size (표 C)

In [15]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../firmware/mcu/simpleserial-haetae
make clean PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 VARIANT=leeha >/dev/null 2>&1
make       PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 VARIANT=leeha 2>&1 | tail -3 && cp simpleserial-haetae-$1.hex haetae-leeha-$1.hex
make clean PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 VARIANT=leeha FAULT_SIM=1 >/dev/null 2>&1
make       PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 VARIANT=leeha FAULT_SIM=1 >/dev/null 2>&1 && cp simpleserial-haetae-$1.hex haetae-leeha-FSIM-$1.hex
echo 'rebuilt leeha (수정판)'
arm-none-eabi-size simpleserial-haetae-$1.elf

+ CRYPTO_TARGET = NONE
+ CRYPTO_OPTIONS = 
+--------------------------------------------------------
rebuilt leeha (수정판)
   text	   data	    bss	    dec	    hex	filename
  31156	    564	  14264	  45984	   b3a0	simpleserial-haetae-CW308_STM32F4.elf


In [16]:
flash('haetae-leeha-FSIM-{}.hex'.format(PLATFORM)); sweep_variant('leeha')
flash('haetae-leeha-{}.hex'.format(PLATFORM)); ss_keygen()
c = ss_cycles(); print('leeha cycles =', c, ' 상대 =', round(c/60858575, 3), 'x')

Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 31723 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 31723 bytes
[leeha]
   NONE     ba9f152c607b207fc6512635ba11388c
   SEED     a70948f5783c58a8596fa54e76dd1e9a
   SIGNBIT  a70948f5783c58a8596fa54e76dd1e9a
   UNPACK   a70948f5783c58a8596fa54e76dd1e9a
   LSB      a70948f5783c58a8596fa54e76dd1e9a
   CS       a70948f5783c58a8596fa54e76dd1e9a
   ADDY     a70948f5783c58a8596fa54e76dd1e9a
   REJECT   ab4a71a7f8084e289653c64b8105a54d
Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 31315 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 31315 bytes
leeha cycles = 69175415  상대 = 1.137 x


## EXP7 (축 A) — 하드웨어 클럭글리치 T2 (랜덤 탐색, AES 검증 스케일)

AES 검증(`Lab_AES_Glitch_Check_F4`)에서 확정: 이 리그 클럭글리치 **정상 동작**, **width sweet spot ≈ 50**(1~12는 너무 작아 실패했던 것).

- **EXP7-a**: 단독 셋업 + 윈도우 `WIN` 측정 + digest기반 `glitch_once`.
- **EXP7-b**: **랜덤 탐색** — width∈(35,70), offset∈(-15,15), ext∈(0,WIN), repeat∈{1,2} 매번 랜덤. `success`=LEAK_T2(`63ff5e..`) → s1 복원.
- **EXP7-c**: 성공 ext 근처 정밀 랜덤 + s1 복원 + 파라미터 맵.

**규칙(디버깅 확정):** hs2='glitch' 전환 후 **리셋** / width·offset **정수** / `scope.capture()` 무시하고 **시리얼 다이제스트로 판정**.

⚠ **HAETAE 전체서명 ~8초/회** → 랜덤 N을 수백으로. +y 히트율이 낮으면, 펌웨어 **트리거를 +y 덧셈 직전으로 좁히는 재빌드**가 최선(윈도우↓ → ext_offset 작게 → 명중률↑). 또는 24MHz 재빌드로 속도 3배↑.

In [1]:
# ===== EXP7-a: 단독 부트스트랩 + '공격지점=트리거지점' 셋업 =====
# 펌웨어 'T' 명령으로 트리거를 특정 연산(기본 ADDY=T2)에만 발생 → 윈도우가 그 연산만 감싸므로
# ext_offset≈0 + width~50 으로 정밀 타격. (리셋 시 g_trig_line=-1로 초기화되므로 매 리셋 후 재설정)
%matplotlib inline
SCOPETYPE='OPENADC'; PLATFORM='CW308_STM32F4'; CRYPTO_TARGET='NONE'; SS_VER='SS_VER_1_1'
import chipwhisperer as cw
import numpy as np, time, struct, csv, collections, logging
logging.getLogger('ChipWhisperer').setLevel(logging.ERROR)

try:
    scope
except NameError:
    scope = cw.scope(name='Husky')
%run "../../Setup_Scripts/Setup_Generic.ipynb"

scope.clock.clkgen_freq = 7.37e6; scope.clock.adc_mul = 1; scope.io.hs2 = 'clkgen'
time.sleep(0.2)

import importlib, haetae_recover; importlib.reload(haetae_recover)
from haetae_recover import attack_recover

FW = '../../../firmware/mcu/simpleserial-haetae/'
FL = {'NONE':0,'SEED':1,'SIGNBIT':2,'UNPACK':3,'LSB':4,'CS':5,'ADDY':6,'REJECT':7}
GOLDEN  = bytes.fromhex('ba9f152c607b207fc6512635ba11388c')
LEAK_T2 = bytes.fromhex('63ff5ebfaa6263739651890939cccb48')
TRIG_POINT = FL['ADDY']       # ★ 공격지점=트리거지점. ADDY=T2(+y). CS=T1, REJECT=RB, 0=전체 c·s~+y

def ss_echo(timeout=3000):
    target.flush(); target.simpleserial_write('e', bytearray())
    r = target.simpleserial_read('r', 16, timeout=timeout); return bytes(r) if r else None
def ss_sign(timeout=90000):
    target.flush(); target.simpleserial_write('p', bytearray([0]*16))
    r = target.simpleserial_read('r', 16, timeout=timeout); return bytes(r) if r else None
def set_fault(line, oneshot=0, checkskip=0):
    target.flush(); target.simpleserial_write('f', bytes([line, oneshot, checkskip]))
    return target.simpleserial_read('r', 1, timeout=3000)
def ss_trig(pt):              # 트리거 지점 설정 (0=전체, FL_*=그 연산만)
    target.flush(); target.simpleserial_write('T', bytes([pt]))
    return target.simpleserial_read('r', 1, timeout=3000)
def flash(hexname):
    cw.program_target(scope, prog, FW + hexname); reset_target(scope); time.sleep(0.5); target.flush()
def recover_target():         # 리셋 후 트리거지점 재설정(리셋이 g_trig_line 초기화하므로)
    reset_target(scope); time.sleep(0.5); target.flush()
    try: ss_trig(TRIG_POINT)
    except Exception: pass

# --- 플래시 + SW결함 OFF + 키준비 + 트리거지점 설정 ---
flash('haetae-baseline-FSIM-{}.hex'.format(PLATFORM))
set_fault(FL['NONE'], 0, 0)
g = ss_sign()                                     # 워밍업(키생성 포함) + GOLDEN 확인
print('echo:', (ss_echo() or b'').hex()[:8], '| sign:', g.hex() if g else None)
assert g == GOLDEN, 'GOLDEN 불일치! (EXP1로 GOLDEN/LEAK_T2 갱신)'
print('trig set:', (ss_trig(TRIG_POINT) or b'').hex(), '(=%02x, 지점 %s)' % (TRIG_POINT, [k for k,v in FL.items() if v==TRIG_POINT][0]))

# (1) 트리거 윈도우 측정 — 이제 TRIG_POINT(+y) 구간만 → 작게 나옴
scope.io.hs2='clkgen'; scope.adc.basic_mode='rising_edge'
try: scope.trigger.triggers='tio4'
except Exception: pass
scope.adc.timeout = 12
scope.arm(); target.simpleserial_write('p', bytearray([0]*16))
to = scope.capture(); WIN = int(scope.adc.trig_count)
target.simpleserial_read('r', 16, timeout=20000)
print('트리거 윈도우 WIN(%s만) = %d 타겟 사이클 | timeout? %s  (수천 이하 기대)'
      % ([k for k,v in FL.items() if v==TRIG_POINT][0], WIN, to))

# (2) 클럭글리치 모드 + 리셋 + 트리거지점 재설정
scope.glitch.enabled = True
scope.glitch.clk_src = 'pll'; scope.glitch.output = 'clock_xor'
scope.glitch.trigger_src = 'ext_single'; scope.glitch.repeat = 1
scope.io.hs2 = 'glitch'; scope.adc.timeout = 3
time.sleep(0.2); reset_target(scope); time.sleep(0.5); target.flush(); ss_trig(TRIG_POINT)
PSS = scope.glitch.phase_shift_steps
print('glitch 모드 | echo:', (ss_echo() or b'').hex()[:8], '| PSS =', PSS)

def set_glitch(delay, width, offset):
    scope.glitch.ext_offset = int(delay); scope.glitch.width = int(width); scope.glitch.offset = int(offset)

def glitch_once(read_timeout=18000):
    scope.arm(); target.flush(); target.simpleserial_write('p', bytearray([0]*16))
    try: scope.capture()
    except Exception: pass
    r = target.simpleserial_read_witherrors('r', 16, glitch_timeout=read_timeout)
    if (not r['valid']) or r['payload'] is None:
        if ss_echo(800) is None: recover_target()
        else: target.flush()
        return 'mute', None
    p = bytes(r['payload'])
    return ('normal', p) if p == GOLDEN else (('success', p) if p == LEAK_T2 else ('other', p))

print('glitch 준비 완료 — WIN 작으면 EXP7-b 랜덤이 +y만 정밀 타격')

INFO: Found ChipWhisperer😍
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen            

(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:704) Target clock may drop; you may need to reset your target.


Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 25895 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 25895 bytes
echo: a0a1a2a3 | sign: ba9f152c607b207fc6512635ba11388c
trig set: 06 (=06, 지점 ADDY)
트리거 윈도우 WIN(ADDY만) = 15568 타겟 사이클 | timeout? False  (수천 이하 기대)
glitch 모드 | echo: a0a1a2a3 | PSS = 4592
glitch 준비 완료 — WIN 작으면 EXP7-b 랜덤이 +y만 정밀 타격


In [ ]:
# EXP7-b: 랜덤 글리치 탐색 — 프로그래스바로 진행 모니터링(그래프는 끝에 1회). success=LEAK_T2 → s1 복원.
import random, collections
from tqdm.notebook import trange
N_TRIES    = 300
W_RANGE    = (35, 70); O_RANGE = (-15, 15); REP_POOL = [1, 1, 2]
E_MAX      = int(WIN * 1.05) if WIN > 0 else 4000     # +y 윈도우(EXP7-a WIN)
STOP_AFTER = 2
cnt = collections.OrderedDict(golden=0, LEAK=0, blocked=0, mute=0); log=[]; leakp=[]
bar = trange(N_TRIES, desc='HAETAE glitch @+y')       # 진행바만 = 빠름(그래프 매회 안 그림)
for i in bar:
    w = random.randint(*W_RANGE); o = random.randint(*O_RANGE); e = random.randint(0, E_MAX); rep = random.choice(REP_POOL)
    scope.glitch.width=int(w); scope.glitch.offset=int(o); scope.glitch.ext_offset=int(e); scope.glitch.repeat=int(rep)
    g, p = glitch_once()
    cls = {'success':'LEAK', 'normal':'golden', 'mute':'mute'}.get(g, 'blocked')
    cnt[cls] += 1; log.append((int(e), int(w), int(o), int(rep), cls, p.hex() if p else ''))
    if cls == 'LEAK':
        leakp.append((e, w, o, rep))
        try:
            r = attack_recover(target); bar.write('★ LEAK ext=%d w=%d o=%d rep=%d → s1 복원율 %.1f%%' % (e, w, o, rep, 100*r['agreement']))
        except Exception: bar.write('★ LEAK ext=%d w=%d o=%d rep=%d' % (e, w, o, rep))
    bar.set_postfix(last='w%d e%d' % (w, e), **cnt)    # 진행상황: 마지막 파라미터 + 누계
    if cnt['LEAK'] >= STOP_AFTER: bar.write('LEAK 확보 — 종료'); break
scope.glitch.repeat = 1
print('DONE:', dict(cnt), '| leak_params:', leakp[:5])
with open('haetae_f4_glitch_log.csv', 'w', newline='') as f:
    wr = csv.writer(f); wr.writerow(['ext','width','offset','repeat','class','digest']); wr.writerows(log)
print('saved haetae_f4_glitch_log.csv')

# --- 파라미터 히트맵: 루프 끝난 뒤 1회만 (진행 속도에 영향 없음) ---
import matplotlib.pyplot as plt
CMAP = collections.OrderedDict(golden='0.75', LEAK='red', blocked='orange', mute='black')
fig, ax = plt.subplots(figsize=(9, 3.6), dpi=120)
for cls, col in CMAP.items():
    pts = [(r[0], r[1]) for r in log if r[4] == cls]
    if pts:
        xs, ys = zip(*pts); ax.scatter(xs, ys, c=col, s=24, alpha=0.75, edgecolors='none', label='%s (%d)' % (cls, cnt[cls]))
ax.set_xlim(0, max(E_MAX, 1)); ax.set_ylim(W_RANGE[0]-2, W_RANGE[1]+2)
ax.set_xlabel('ext_offset (cycles in +y window)'); ax.set_ylabel('glitch width')
ax.set_title('HAETAE HW clock-glitch @+y  (red = LEAK / T2 key leak)'); ax.legend(loc='upper right', fontsize=8)
fig.tight_layout(); fig.savefig('fig_haetae_glitch_map.png'); fig.savefig('fig_haetae_glitch_map.pdf'); plt.show()
print('saved fig_haetae_glitch_map.{png,pdf}')

In [ ]:
# EXP7-b2 (진단): +y에서 width 미세 스윕 — clean-fault 밴드 탐색
# (EXP7-a 실행 후 사용. WIN·glitch_once 필요. 전부 mute면 width가 과한 것 → 밴드를 찾는다.)
import collections
mid = max(WIN // 2, 1)                        # +y 윈도우 중간 지점
cnt = collections.OrderedDict(golden=0, LEAK=0, blocked=0, mute=0)
print('width 0..70 @ +y (ext=%d, offset=0) — 밴드 탐색' % mid, flush=True)
for w in range(0, 71, 2):                     # 0(무글리치 sanity) → 70
    scope.glitch.width = int(w); scope.glitch.offset = 0
    scope.glitch.ext_offset = int(mid); scope.glitch.repeat = 1
    g, p = glitch_once()
    cls = {'success':'LEAK', 'normal':'golden', 'mute':'mute'}.get(g, 'blocked')
    cnt[cls] += 1
    print('w=%2d -> %-8s %s' % (w, cls, p.hex()[:12] if p else ''), flush=True)
print('요약:', dict(cnt))
print('해석: golden(저width)→blocked/LEAK→mute(고width) 전이의 blocked/LEAK 구간이 clean-fault 밴드.')
print('      전부 mute면 +y가 무조건 크래시 → TRIG_POINT=FL[CS]/FL[REJECT] 또는 전압글리치 검토.')

width 0..70 @ +y (ext=7784, offset=0) — 밴드 탐색


In [ ]:
# EXP7-c: LEAK 근처 정밀 랜덤 + s1 복원 + 파라미터 맵 (EXP7-b의 log 사용)
import random, collections
import matplotlib.pyplot as plt
from tqdm.notebook import trange
try: log                                    # EXP7-b가 생성. 없으면 빈 리스트로 시작(단독 실행 대비)
except NameError: log = []
# log 각 행 = (ext, width, offset, repeat, cls, digest), cls∈{golden,LEAK,blocked,mute}
leaks  = [r for r in log if len(r) >= 5 and r[4] == 'LEAK']
base_e = leaks[0][0] if leaks else int(WIN * 0.5)
print('정밀화 중심 ext =', base_e, '| LEAK %d개 / log %d행' % (len(leaks), len(log)), flush=True)
cnt = collections.OrderedDict(golden=0, LEAK=0, blocked=0, mute=0); best=None
for _ in trange(120, desc='refine (random near base_e)'):
    e = max(base_e + random.randint(-800, 800), 0)
    w = random.randint(35, 65); o = random.randint(-12, 12); rep = random.choice([1, 2])
    scope.glitch.width=int(w); scope.glitch.offset=int(o); scope.glitch.ext_offset=int(e); scope.glitch.repeat=int(rep)
    g, p = glitch_once()
    cls = {'success':'LEAK', 'normal':'golden', 'mute':'mute'}.get(g, 'blocked')
    cnt[cls] += 1; log.append((int(e), int(w), int(o), int(rep), cls, p.hex() if p else ''))
    if cls == 'LEAK':
        r = attack_recover(target)
        print('★ s1 복원율 %.1f%%  (ext=%d w=%d o=%d rep=%d)' % (100*r['agreement'], e, w, o, rep))
        if best is None: best = (int(e), int(w), int(o), int(rep))
scope.glitch.repeat = 1
print('done:', dict(cnt), '| best LEAK (ext,w,o,rep):', best)
with open('haetae_f4_glitch_log.csv', 'w', newline='') as f:
    wr = csv.writer(f); wr.writerow(['ext','width','offset','repeat','class','digest']); wr.writerows(log)

# 파라미터 맵 (ext_offset × width): LEAK=빨강+, blocked=회색o
pts = [(r[0], r[1], r[4]) for r in log if len(r) >= 5 and r[4] in ('LEAK', 'blocked')]
fig, ax = plt.subplots(figsize=(8, 3.4), dpi=130)
for e, w, c_ in pts:
    if c_ == 'LEAK': ax.scatter(e, w, c='red', marker='+', s=120, linewidths=2)
    else: ax.scatter(e, w, facecolors='none', edgecolors='gray', s=25)
ax.set_xlabel('ext_offset (target cycles)'); ax.set_ylabel('glitch width')
ax.set_title('HAETAE HW clock-glitch map (red + = T2 LEAK / key leak)')
fig.tight_layout(); fig.savefig('fig_haetae_glitch_map.png'); fig.savefig('fig_haetae_glitch_map.pdf'); plt.show()
print('saved fig_haetae_glitch_map.{png,pdf} | 표시점:', len(pts))

## EXP7-d — 축 A 변형 비교 (실제 클럭글리치 @ +y, 4변형)

EXP2(SW주입 커버리지)의 **하드웨어 버전**. 트리거를 +y(ADDY)에만 두고 baseline/double/leeha/irv에 **동일 랜덤 글리치**(width~50) 주입:
- `LEAK`(=LEAK_T2 `63ff5e..`) → **키 누설**. baseline만 기대.
- `golden`(무영향) / `blocked`(대응기법이 무효화·난수화) / `mute`(크래시).

기대 결과: **baseline은 LEAK 발생, double/leeha/irv는 LEAK=0**(실제 글리치로도 차단). ⚠ 8초/서명(double은 2배) → 변형당 N을 작게.
전제: EXP7-a를 `TRIG_POINT=FL['ADDY']`로 실행해 `WIN`(+y 윈도우)이 잡혀 있어야 함.

In [ ]:
# EXP7-d: 축 A 변형 비교 — 트리거 @+y, 4변형에 동일 랜덤 글리치 (+ 비교 막대그래프)
import random, collections
import matplotlib.pyplot as plt
from tqdm.notebook import trange
assert TRIG_POINT == FL['ADDY'], 'EXP7-a에서 TRIG_POINT=FL[ADDY]로 실행 필요'
N_PER    = 60                       # 변형당 시도 (8s/서명, double은 2배) — 조정 가능
W_RANGE  = (35, 70); O_RANGE = (-15, 15); REP_POOL = [1, 1, 2]
E_MAX    = int(WIN * 1.05) if WIN > 0 else 4000
VARIANTS = ['baseline', 'double', 'leeha', 'irv']
CLASSES  = ['golden', 'LEAK', 'blocked', 'mute']
CCOL     = {'golden':'0.75', 'LEAK':'red', 'blocked':'orange', 'mute':'black'}
summary = collections.OrderedDict(); glog = []

for V in VARIANTS:
    scope.io.hs2 = 'clkgen'
    flash('haetae-{}-FSIM-{}.hex'.format(V, PLATFORM))
    set_fault(FL['NONE'], 0, 0)
    scope.io.hs2 = 'glitch'; scope.adc.timeout = 3
    time.sleep(0.2); reset_target(scope); time.sleep(0.5); target.flush(); ss_trig(TRIG_POINT)
    if ss_sign() is None: recover_target(); ss_sign()
    cnt = collections.OrderedDict(golden=0, LEAK=0, blocked=0, mute=0); leakp = []
    bar = trange(N_PER, desc='[%s] @+y' % V)
    for i in bar:
        w = random.randint(*W_RANGE); o = random.randint(*O_RANGE); e = random.randint(0, E_MAX); rep = random.choice(REP_POOL)
        scope.glitch.width=int(w); scope.glitch.offset=int(o); scope.glitch.ext_offset=int(e); scope.glitch.repeat=int(rep)
        g, p = glitch_once()
        cls = {'success':'LEAK', 'normal':'golden', 'mute':'mute'}.get(g, 'blocked')
        cnt[cls] += 1; glog.append((V, e, w, o, rep, cls, p.hex() if p else ''))
        if cls == 'LEAK': leakp.append((e, w, o, rep))
        bar.set_postfix(**cnt)
        if cls == 'LEAK' and V == 'baseline' and len(leakp) == 1:
            try: r = attack_recover(target); bar.write('  baseline LEAK → s1 복원율 %.1f%%' % (100*r['agreement']))
            except Exception: pass
    scope.glitch.repeat = 1
    summary[V] = dict(cnt); print('[%s] %s | leak_params=%s' % (V, dict(cnt), leakp[:3]), flush=True)

print('\n===== 축 A HW 클럭글리치 @+y — 변형 비교 =====')
print('%-9s %6s %8s %8s %6s' % ('variant', 'LEAK', 'golden', 'blocked', 'mute'))
for V, s in summary.items():
    print('%-9s %6d %8d %8d %6d' % (V, s['LEAK'], s['golden'], s['blocked'], s['mute']))
print('\n해석: baseline만 LEAK(=LEAK_T2, s1 누설). double/leeha/irv는 LEAK=0이면 실제 글리치로도 차단 입증.')

# --- 비교 막대그래프 (변형별 결과 누적) ---
fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(VARIANTS)); bottom = [0]*len(VARIANTS)
for cls in CLASSES:
    vals = [summary[V][cls] for V in VARIANTS]
    ax.bar(x, vals, bottom=bottom, color=CCOL[cls], label=cls, edgecolor='white')
    for xi, (vv, bb) in enumerate(zip(vals, bottom)):
        if vv: ax.text(xi, bb + vv/2, str(vv), ha='center', va='center',
                       color=('white' if cls in ('LEAK','mute') else 'black'), fontsize=8)
    bottom = [b + v for b, v in zip(bottom, vals)]
ax.set_xticks(list(x)); ax.set_xticklabels(VARIANTS)
ax.set_ylabel('count (N=%d/variant)' % N_PER); ax.legend(loc='upper right', fontsize=8)
ax.set_title('Axis-A HW clock-glitch @+y : baseline leaks, countermeasures block')
fig.tight_layout(); fig.savefig('fig_axisA_variants.png', dpi=130); fig.savefig('fig_axisA_variants.pdf'); plt.show()

with open('haetae_f4_axisA_variants.csv', 'w', newline='') as f:
    wr = csv.writer(f); wr.writerow(['variant','LEAK','golden','blocked','mute'])
    for V, s in summary.items(): wr.writerow([V, s['LEAK'], s['golden'], s['blocked'], s['mute']])
    wr.writerow([]); wr.writerow(['variant','ext','width','offset','repeat','class','digest']); wr.writerows(glog)
print('saved haetae_f4_axisA_variants.csv + fig_axisA_variants.{png,pdf}')